# The Project Design

## 1. Perception Layer (Image Processing)

These components process raw user photos to detect clothing items and extract attributes. Where supported by the platform, models can run in parallel using threads.

### A. Clothing Detection & Segmentation (RF-DETR Segmentation Nano)

**Model:** RF-DETR Segmentation Nano (Instance Segmentation), trained on Fashionpedia (≈50K images with masks).
**Purpose:** Detect garments (top/bottom/shoes), crop them, and provide category + confidence.

#### Input:

A raw photo of a person or a flat-lay clothing image.

#### Output:

A list of detected clothing items (bbox + mask + category + confidence + cropped image).

#### Example:

```txt
Input:
    PIL.Image()  # e.g. (1920x1080)

Output:
    [
        {
            "category_id": 1,         # "Top"
            "confidence": 0.98,
            "bbox": [100, 50, 400, 500],
            "mask": np.ndarray(...),  # optional binary mask
            "image": PIL.Image()      # Cropped Top
        },
        {
            "category_id": 2,         # "Bottom"
            "confidence": 0.94,
            "bbox": [120, 420, 420, 980],
            "mask": np.ndarray(...),
            "image": PIL.Image()
        }
    ]
```

In [ ]:
import os
from PIL import Image
from matplotlib import pyplot as plt

from perception_layer.clothing_detection_yolo_backup import model, inference

# example_imgs = [Image.open(f"examples/{path}") for path in os.listdir("examples")]
example_imgs = [Image.open("examples/ex6.jpeg")]
# model_path = "perception_layer/clothing_detection_segmintation/rf_detr_fasionopedia/checkpoint_best_regular.pth"
garment_datector = inference.GarmentDetector()
outfits = {}

for img in example_imgs:
    garments = garment_datector.predict(img, threshold=0.40)
    inference.plot_wardrobe_detections(img, garments)
    print(f"Detected {len(garments)} garments in {img.filename}")
    for garment in garments:
        print(f"  {garment['category_name']} ({garment['confidence']:.2f})")
        plt.imshow(garment["image"])
    outfits[img.filename] = {"garments":garments}

outfits

In [ ]:
outfits[example_imgs[0].filename]["garments"][6]["image"]

In [ ]:
example_imgs[0].filename

### B. Multi-Attribute Classifier (EfficientNet-B0, Multi-Head)

**Architecture:** Single backbone + multiple heads.
**Why:** Run one forward pass and predict multiple attributes (saves compute vs separate models).

**Predicted attributes (examples):**

* fit: slim / regular / oversized
* style: formal / casual / sporty
* weather_warmth: 0–1 (cold → warm)
* formality_score: 0–1

#### Input:

A normalized cropped item image, optionally with the garment type id.

#### Output:

Attribute probabilities + scalar metrics.

#### Example:

```txt
Input:
    torch.Tensor()  # Shape: (1, 3, 224, 224)

Output:
    {
        "fit": torch.Tensor([0.1, 0.8, 0.1]),            # -> "Regular"
        "style": torch.Tensor([0.05, 0.9, 0.05]),        # -> "Casual"
        "weather_warmth": 0.75,                          # 0.0 cold -> 1.0 warm
        "formality_score": 0.20                          # 0.0 PJs  -> 1.0 tux
    }
```
**Labeling strategy:**

* Cloud training can use **pseudo-labeling** via a vision-language model (CLIP-style) + a small manually verified set.

---

In [ ]:
from perception_layer.multi_attribute_classifire import inference
model_path = "perception_layer/multi_attribute_classifire/runs2/1b/best_model.pt"
multi_attribute_classifier = inference.AttributePredictor(model_path)
for img in example_imgs:
    for idx, garment in enumerate(outfits[img.filename]["garments"]):
        img_att = multi_attribute_classifier.predict(garment["image"].convert("RGB"))
        outfits[img.filename]["garments"][idx]["attributes"] = img_att
    print(outfits[img.filename])

In [ ]:
outfits["examples/me.jpeg"]["garments"][4]["attributes"]

In [ ]:
outfits[example_imgs[0].filename]["garments"]

### E. Color Harmony Score (Function)

#### Input:

Cropped clothing item images and the number of dominant colors per garment.

#### Output:

Color harmony score between 0 and 1 of the cropped images combined

#### Example:
```txt
Input:
    images : list[PIL.Image, PIL.Image, PIL.Image] # 3 images
    k : 5 # number of dominant colors per garment

Output:
    0.75 #  The color harmony score is 75%
```

In [ ]:
# from perception_layer import color_utils
# for img in example_imgs:
#     outfit_cropped_imgs = [garment["image"].convert("RGB") for garment in outfits[img.filename]["garments"]]
#     outfits[img.filename]["color_harmony"] = color_utils.harmony_score_from_images(outfit_cropped_imgs)
#     print(outfits[img.filename])

## 2. Textual Query Understanding (Semantic Processing)

This layer transforms a user request (text) into intent signals and an embedding usable by ranking.

### A. Query Vectorization (Mobile Text Encoder)

**Model:** Distilled BERT / MobileCLIP text branch.
**Purpose:** Encode user query into a 512-dim vector aligned with image vectors.

#### Input:

A user query string.

#### Output:

A 512-dim embedding vector.

#### Example:

```txt
Input:
    "business meeting, looking for comfort"

Output:
    torch.Tensor()  # Shape: (1, 512)
```

In [ ]:
from semantic_processing.ranking_utils import garment_score
from semantic_processing.query_vectorization.query_vectorizer import QueryVectorizer

usr_input_sentence = "I have a business meeting, looking for comfort"
sentence_transformer = QueryVectorizer(model_path="models/nlp_query/distilled_query_encoder.pth")
query_vec = sentence_transformer.encode(usr_input_sentence)

filters = {
    "min_formality": 0.4,
    "max_warmth": 0.7,
    "sporty_ok": False
}

context = {
    "temperature": 20
}
for filename, data in outfits.items():
    for garment in data["garments"]:
        score = garment_score(
            garment,
            query_vec,
            filters,
            context
        )

        garment["match_score"] = round(score, 3)

    print(filename, data["garments"])

In [ ]:
usr_input_sentence = "I have a business meeting, looking for comfort"
sentence_transformer = QueryVectorizer(model_path="models/nlp_query/distilled_query_encoder.pth")
query_vec = sentence_transformer.encode(usr_input_sentence)
query_vec[0][:5], query_vec.shape

## 3. Representation Layer (Vectorization)

This layer produces vectors for fast similarity search and outfit relevance.

### A. Visual Embeddings (Student CLIP)

**Model:** MobileNetV3 image encoder distilled from CLIP (ViT teacher).
**Purpose:** Convert each item image into a normalized 512-dim “vibe” embedding.

#### Input:

Cropped image tensor.

#### Output:

Normalized embedding.

#### Example:

```txt
Input:
    torch.Tensor()  # (1, 3, 224, 224)

Output:
    torch.Tensor()  # (1, 512)
```

---

### B. Text Embeddings

**Purpose:** Ensure text and image vectors live in the same embedding space (512-dim).

---

### C. Local Vector Store

**Storage:** SQLite + local files, or a compact ANN index.
**Purpose:** Nearest-neighbor retrieval without internet.

#### Example:

```txt
Stored:
    item_id -> embedding(512) + metadata
Search:
    top_k by cosine similarity
```

---

In [ ]:
garment_img = outfits[example_imgs[0].filename]["garments"][4]["image"]

In [ ]:
from representation_layer.visual_embeddings.inference import GarmentEmbedder

img_embedder = GarmentEmbedder("models/visual_embedder/best_student_model.pth")
img_embedding = img_embedder.embed_crop(garment_img)

for img in example_imgs:
    for idx, garment in enumerate(outfits[img.filename]["garments"]):
        img_vec = img_embedder.embed_crop(garment["image"].convert("RGB"))
        outfits[img.filename]["garments"][idx]["img_vec"] = img_vec
    print(outfits[img.filename])

In [ ]:
from recomendation_engine import demo_engine
engine = demo_engine.FashionBrain()

final_score, outfit_emb, s_style, s_rel = engine.score_outfit(outfits[example_imgs[0].filename], query_vec)
final_score, outfit_emb.shape

In [ ]:
engine.update_feedback(
    outfit_emb=outfit_emb,
    liked=True,
    s_style=s_style,
    s_rel=s_rel
)

In [ ]:
final_score, outfit_emb, s_style, s_rel = engine.score_outfit(outfits[example_imgs[0].filename], query_vec)
final_score

In [ ]:
engine.score_outfit(outfits[example_imgs[0].filename], query_vec)

In [ ]:
outfits[example_imgs[0].filename]